# Flip Story

## Research Question

### In how many constituencies did the winning party change between 2021 and 2026?

The Geographic Story showed where political power shifted across Tamil Nadu.

The Flip Story examines how constituencies changed hands between parties and identifies the major pathways through which political power was redistributed.

---

## Analysis Roadmap

1. Load Data
2. Identify Winners
3. Track Constituency-Level Changes
4. Measure Seat Flips
5. Analyze Flip Directions
6. Create Sankey Diagram
7. Flip Story Conclusion

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
results_2021 = pd.read_csv("../data/tn_2021_results.csv")
results_2026 = pd.read_csv("../data/tn_2026_results.csv")

# Identify Constituency Winners

The election data contains one row per candidate.

To analyze seat flips, we first identify the winning candidate in each constituency for both elections.

In [3]:
winner2021 = (
    results_2021
    .sort_values('votes', ascending=False)
    .groupby('ac_number')
    .first()
    .reset_index()
)

winner2021.head()

,ac_number,constituency,candidate,party,votes,turnout,reserved,region
0,1,Gummidipundi,GOVINDARAJAN T.J,DMK,126452,78.08,GEN,Chennai Metro
1,2,Ponneri,DURAI. CHANDRASEKAR,INC,94528,78.20,SC,Chennai Metro
2,3,Tiruttani,S.Chandran,DMK,120314,78.76,GEN,Chennai Metro
3,4,Thiruvallur,"RAAJENDRAN, V.G.",DMK,107709,77.20,GEN,Chennai Metro
4,5,Poonamallee,Krishnaswamy A,DMK,149578,72.99,SC,Chennai Metro


In [4]:
winner2026 = (
    results_2026
    .sort_values('votes', ascending=False)
    .groupby('ac_number')
    .first()
    .reset_index()
)

winner2026.head()

,ac_number,constituency,candidate,party,votes,turnout,reserved,region
0,1,Gummidipoondi,S.VIJAYAKUMAR,TVK,94320,NaN,GEN,Chennai Metro
1,2,Ponneri,DR.RAVI.M.S,TVK,110439,NaN,SC,Chennai Metro
2,3,Tiruttani,G.HARI,AIADMK,89169,NaN,GEN,Chennai Metro
3,4,Thiruvallur,DR. T. ARUNKUMAR,TVK,92190,NaN,GEN,Chennai Metro
4,5,Poonamallee,PRAKASAM.R,TVK,161309,NaN,SC,Chennai Metro


In [5]:
winner2021['party_group'] = winner2021['party'].apply(
    lambda x: x if x in ['DMK', 'AIADMK', 'TVK'] else 'Others'
)

winner2026['party_group'] = winner2026['party'].apply(
    lambda x: x if x in ['DMK', 'AIADMK', 'TVK'] else 'Others'
)

# Track Constituency-Level Changes

To understand seat flips, we compare the winning party in every constituency between 2021 and 2026.

In [6]:
flip = winner2021[
    ['ac_number', 'constituency', 'party_group']
].merge(
    winner2026[
        ['ac_number', 'party_group']
    ],
    on='ac_number',
    suffixes=('_2021', '_2026')
)

flip.head()

,ac_number,constituency,party_group_2021,party_group_2026
0,1,Gummidipundi,DMK,TVK
1,2,Ponneri,Others,TVK
2,3,Tiruttani,DMK,AIADMK
3,4,Thiruvallur,DMK,TVK
4,5,Poonamallee,DMK,TVK


In [7]:
flip['status'] = np.where(
    flip['party_group_2021']
    ==
    flip['party_group_2026'],
    'Retained',
    'Flipped'
)

flip.head()

,ac_number,constituency,party_group_2021,party_group_2026,status
0,1,Gummidipundi,DMK,TVK,Flipped
1,2,Ponneri,Others,TVK,Flipped
2,3,Tiruttani,DMK,AIADMK,Flipped
3,4,Thiruvallur,DMK,TVK,Flipped
4,5,Poonamallee,DMK,TVK,Flipped


In [8]:
flip['status'].value_counts()

status
Flipped     161
Retained     73
Name: count, dtype: int64

In [9]:
total_seats = len(flip)

flipped_seats = (
    flip['status']
    .eq('Flipped')
    .sum()
)

retained_seats = (
    flip['status']
    .eq('Retained')
    .sum()
)

flip_pct = round(
    flipped_seats / total_seats * 100,
    1
)

print("Total Seats :", total_seats)
print("Flipped Seats :", flipped_seats)
print("Retained Seats :", retained_seats)
print("Flip Percentage :", flip_pct, "%")

Total Seats : 234
Flipped Seats : 161
Retained Seats : 73
Flip Percentage : 68.8 %


In [10]:
flip['status'].value_counts()

status
Flipped     161
Retained     73
Name: count, dtype: int64

In [11]:
print("Flip Percentage :", flip_pct, "%")

Flip Percentage : 68.8 %


## Scale of Change

### Key Observations

- 161 of 234 constituencies changed hands between 2021 and 2026.
- Only 73 constituencies retained the same winning formation.
- Nearly 7 out of every 10 seats (68.8%) experienced a change in winning party.
- The scale of constituency-level change indicates that the 2026 election was not a marginal shift but a significant restructuring of political representation across Tamil Nadu.

# Analyze Flip Directions

Not all flipped constituencies moved in the same direction.

To understand the redistribution of political power, we analyze how seats moved between formations.

In [12]:
flip_direction = (
    flip[flip['status'] == 'Flipped']
    .groupby(
        ['party_group_2021', 'party_group_2026']
    )
    .size()
    .reset_index(name='total_seats')
)

flip_direction

,party_group_2021,party_group_2026,total_seats
0,AIADMK,DMK,15
1,AIADMK,Others,3
2,AIADMK,TVK,26
3,DMK,AIADMK,22
4,DMK,Others,6
5,DMK,TVK,65
6,Others,AIADMK,3
7,Others,DMK,4
8,Others,TVK,17


In [14]:
flip_direction['flip_direction'] = (
    flip_direction['party_group_2021']
    + ' → ' +
    flip_direction['party_group_2026']
)

flip_direction

,party_group_2021,party_group_2026,total_seats,flip_direction
0,AIADMK,DMK,15,AIADMK → DMK
1,AIADMK,Others,3,AIADMK → Others
2,AIADMK,TVK,26,AIADMK → TVK
3,DMK,AIADMK,22,DMK → AIADMK
4,DMK,Others,6,DMK → Others
5,DMK,TVK,65,DMK → TVK
6,Others,AIADMK,3,Others → AIADMK
7,Others,DMK,4,Others → DMK
8,Others,TVK,17,Others → TVK
